# Weight-space programming, step 2: hand-edited JEPA adapters on v1's 52 cases

The probes (`jepa-adapter-weight-probe`) located the noise-keeping effect of the JEPA latents in the adapter's `glob` path, the offset it adds to ADM's time embedding. 87 % of that offset is **the same vector for every image** (a 'conditioning on' mode learned because 90 % of training samples had latents and 10 % had none), and only 0.05 % of it lies along the time-embedding tangent, so it is not a plain timestep shift. Two edits, written by reasoning over the weights (no training, no gradients); both leave the null-latent output exactly unchanged:

| edit | change | weights touched |
|---|---|---|
| **E2** | `glob.0.weight = 0`: the `glob` path ignores the latents; the spatial branches keep them | 1.31 M (zeroed) |
| **E3** | `glob.2.weight -= m vᵀ`: m = mean latent-induced offset over the 80 probe cases, v with v·h(0) = 0 and v·h(c) ≈ 1 (ridge LS, λ = 0.01); removes the common offset, keeps the image-specific part | 1.05 M (rank-1) |

Every case runs ADM, the original adapter (null and JEPA latents) and both edits in one batch, with the same start and noise draws, on v1's inputs and seeds (checked against its `scores.csv`).

Kaggle: GPU T4 ×2, internet on, outputs of `jepa-denoise-finetune` (v2), `jepa-guided-diffusion` (v1) and `jepa-adapter-weight-probe` attached.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips"], check=True)

In [ ]:
import copy, glob, io, json, math, os, random, re, subprocess, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.checkpoint import checkpoint
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

N_EVAL, SIGMAS = 16, [0.1, 0.2, 0.3]
DIFFUSION_IMAGES = [0, 1, 2, 3, 9]   # cases whose outputs are saved as PNG (all real pairs too)
E3_RIDGE = 0.01

assert torch.cuda.is_available(), "GPU required"
D0 = torch.device("cuda:0")
D1 = torch.device("cuda:1") if torch.cuda.device_count() > 1 else D0
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], torch.__version__)
CACHE, OUT = os.environ.get("CACHE_DIR", "/tmp/guided_cache"), "outputs"
INPUT_ROOTS = ["/kaggle/input", os.environ.get("INPUT_DIR", "/workspace/inputs")]
for d in (CACHE, OUT, f"{OUT}/images"):
    os.makedirs(d, exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
S = 256
TIMES = {}


def find_input(pattern):
    """First file matching the glob pattern under the input roots (Kaggle-attached outputs or $INPUT_DIR), else None."""
    for root in INPUT_ROOTS:
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return hits[0]
    return None


def download(url, path, size=None):
    """Resumable download with progress; no-op when the file is already complete."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    if size and have == size:
        return path
    if size and have > size:
        have = 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if have and r.status != 206:
            have = 0  # server ignored the Range header: start over
        total, t0, got, mark = have + int(r.headers.get("Content-Length", 0)), time.time(), 0, 0
        with open(path, "ab" if have else "wb") as f:
            while chunk := r.read(1 << 23):
                f.write(chunk)
                got += len(chunk)
                if got - mark >= 1 << 30:
                    mark = got
                    print(f"  {(have + got) / 1e9:.1f}/{total / 1e9:.1f} GB  {got / 1e6 / (time.time() - t0):.0f} MB/s", flush=True)
    if size:
        assert os.path.getsize(path) == size, f"incomplete download: {path}"
    return path


def load_zip_images(path, short):
    with zipfile.ZipFile(path) as z:
        names = sorted(n for n in z.namelist() if n.lower().endswith(".png"))

    def one(name):
        with zipfile.ZipFile(path) as z:
            im = Image.open(io.BytesIO(z.read(name))).convert("RGB")
        return torch.from_numpy(np.asarray(TF.resize(im, short)).copy()).permute(2, 0, 1).contiguous()

    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(one, names))


def fetch_image(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")


def add_gaussian(x, sigma, seed=0):
    return (x + sigma * torch.randn(x.shape, generator=torch.Generator().manual_seed(seed))).clamp(0, 1)


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def save_png(img, path):
    Image.fromarray((img.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)).save(path)


V1_SCORES = find_input("jepa-guided-diffusion/**/scores.csv")
assert V1_SCORES, "attach the output of the jepa-guided-diffusion notebook (version 1) as input data"
ORIGINAL = pd.read_csv(V1_SCORES, keep_default_na=False)

## Inputs: identical to v1's evaluation

In [ ]:
t0 = time.time()
DIV2K = "https://data.vision.ee.ethz.ch/cvl/DIV2K/"
VALID_IMGS = load_zip_images(download(DIV2K + "DIV2K_valid_HR.zip", f"{CACHE}/DIV2K_valid_HR.zip", size=448_993_893), S)
EVAL_CLEAN = torch.stack([TF.center_crop(im, S).float() / 255 for im in VALID_IMGS[:N_EVAL]])
POLYU = "https://raw.githubusercontent.com/csjunxu/PolyU-Real-World-Noisy-Images-Dataset/master/CroppedImages/"
REAL = []
for n in ["Canon80D_8_8_3200_ball_1", "Canon5D2_5_160_6400_bicycle_10", "NikonD800_5_100_4000_flower_15", "Sony_4-5_125_3200_plant_13"]:
    noisy, clean = (TF.to_tensor(TF.center_crop(fetch_image(f"{POLYU}{n}_{k}.JPG"), S)) for k in ("real", "mean"))
    REAL.append((n, noisy, clean))

CASES = [dict(image=f"div2k_{i}", noise=f"gaussian {s}", sigma=s, clean=y, noisy=add_gaussian(y, s, seed=1000 * i + j))
         for i, y in enumerate(EVAL_CLEAN) for j, s in enumerate(SIGMAS)]
CASES += [dict(image=n, noise="real camera", sigma=float((noisy - clean).std()), clean=clean, noisy=noisy) for n, noisy, clean in REAL]
DIFFUSION_KEYS = {f"div2k_{i}" for i in DIFFUSION_IMAGES}
for k, case in enumerate(CASES):
    case["seed"] = k  # denoise_variants(..., seed=k) in the original notebook
    case["save"] = case["noise"] == "real camera" or case["image"] in DIFFUSION_KEYS

noisy_psnr = ORIGINAL[ORIGINAL.method == "noisy input"].set_index(["image", "noise"]).psnr
INPUT_GAP = max(abs(peak_signal_noise_ratio(c["clean"].permute(1, 2, 0).numpy(), c["noisy"].permute(1, 2, 0).numpy(), data_range=1.0)
                    - noisy_psnr[(c["image"], c["noise"])]) for c in CASES)
print(f"{len(CASES)} cases, {sum(c['save'] for c in CASES)} saved as PNG; "
      f"max |noisy-input PSNR - original run| = {INPUT_GAP:.1e} dB")
assert INPUT_GAP < 1e-3, "the inputs differ from the guided-diffusion run"
TIMES["inputs"] = time.time() - t0

In [ ]:
GD, IJ = f"{CACHE}/guided-diffusion", f"{CACHE}/ijepa"
for url, path in (("https://github.com/openai/guided-diffusion", GD), ("https://github.com/facebookresearch/ijepa", IJ)):
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, path], check=True)
sys.path[:0] = [GD, IJ]
from guided_diffusion.script_util import create_model
from guided_diffusion.nn import timestep_embedding
import guided_diffusion.unet as gd_unet

# ADM's attention blocks always use its own checkpoint function, which differentiates w.r.t. the block weights and
# fails when they are frozen; eps_pred() checkpoints whole blocks with torch.utils.checkpoint instead.
gd_unet.checkpoint = lambda func, inputs, params, flag: func(*inputs)
from src.models import vision_transformer as vt

adm = create_model(image_size=256, num_channels=256, num_res_blocks=2, learn_sigma=True, class_cond=False,
                   attention_resolutions="32,16,8", num_head_channels=64, resblock_updown=True, use_fp16=True,
                   use_scale_shift_norm=True)  # the README's flags for 256x256_diffusion_uncond.pt
ADM_CKPT = download("https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt",
                    f"{CACHE}/256x256_diffusion_uncond.pt", size=2_211_383_297)
adm.load_state_dict(torch.load(ADM_CKPT, map_location="cpu", weights_only=True))
adm.to(D0).eval().requires_grad_(False).convert_to_fp16()


def _sdpa_attention(self, x):  # same math as I-JEPA's attention, memory-efficient kernel
    B, n, C = x.shape
    q, k, v = self.qkv(x).reshape(B, n, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
    x = F.scaled_dot_product_attention(q, k, v, scale=self.scale)
    return self.proj(x.transpose(1, 2).reshape(B, n, C)), None


vt.Attention.forward = _sdpa_attention
found = [path for path in [find_input("jepa_denoise_vith14_fp16.pt")] if path]
assert found, "attach the output of the jepa-denoise-finetune notebook (version 2) as input data"
jepa = vt.vit_huge(patch_size=14)
jepa.load_state_dict(torch.load(found[0], map_location="cpu", weights_only=True)["target_encoder"])
jepa = jepa.half().to(D1).eval().requires_grad_(False)
JMEAN = torch.tensor([0.485, 0.456, 0.406], device=D1).view(1, 3, 1, 1)
JSTD = torch.tensor([0.229, 0.224, 0.225], device=D1).view(1, 3, 1, 1)


@torch.no_grad()
def jepa_latents(x):
    """x [B,3,S,S] in [0,1] -> [B,D,16,16]: LayerNormed target-encoder tokens of the 224 px resized image."""
    x = F.interpolate(x.to(D1).float(), size=(224, 224), mode="bilinear", antialias=True, align_corners=False)
    with torch.autocast("cuda", dtype=torch.float16):
        h = jepa((x - JMEAN) / JSTD)
    h = F.layer_norm(h.float(), (h.size(-1),))
    return h.transpose(1, 2).reshape(len(x), h.size(-1), 16, 16).to(D0)

In [ ]:
@torch.no_grad()
def injection_points(res_list=(32, 16, 8)):
    """{ADM encoder block index: (resolution, channels)} for the first block at each resolution."""
    emb = adm.time_embed(timestep_embedding(torch.zeros(1, dtype=torch.long, device=D0), adm.model_channels))
    h, points = torch.zeros(1, 3, S, S, device=D0).type(adm.dtype), {}
    for i, m in enumerate(adm.input_blocks):
        h = m(h, emb)
        if h.shape[-1] in res_list and h.shape[-1] not in [r for r, _ in points.values()]:
            points[i] = (h.shape[-1], h.shape[1])
    return points


def _zero(m):
    nn.init.zeros_(m.weight)
    nn.init.zeros_(m.bias)
    return m


class JEPAAdapter(nn.Module):
    """JEPA latents [B,D,16,16] -> features added to ADM encoder blocks + an offset on ADM's time embedding."""

    def __init__(self, d, points, emb_dim):
        super().__init__()
        self.res = {str(i): r for i, (r, _) in points.items()}
        self.branches = nn.ModuleDict({str(i): nn.Sequential(
            nn.Conv2d(d, ch, 1), nn.GroupNorm(32, ch), nn.SiLU(), nn.Conv2d(ch, ch, 3, padding=1), nn.SiLU(),
            _zero(nn.Conv2d(ch, ch, 1))) for i, (r, ch) in points.items()})
        self.glob = nn.Sequential(nn.Linear(d, emb_dim), nn.SiLU(), _zero(nn.Linear(emb_dim, emb_dim)))

    def forward(self, c):
        feats = {}
        for i, branch in self.branches.items():
            r = self.res[i]
            feats[int(i)] = branch(c if r == c.shape[-1] else F.interpolate(c, size=(r, r), mode="bilinear", align_corners=False))
        return feats, self.glob(c.mean((2, 3)))


def eps_pred(x, t, cond=None, grad_ckpt=False):
    """ADM's UNetModel.forward (ε channels only) with adapter features added after the injection blocks."""
    emb = adm.time_embed(timestep_embedding(t, adm.model_channels))
    feats = {}
    if cond is not None:
        feats, g = cond
        emb = emb + g
    run = (lambda m, *a: checkpoint(m, *a, use_reentrant=False)) if grad_ckpt else (lambda m, *a: m(*a))
    hs, h = [], x.type(adm.dtype)
    for i, m in enumerate(adm.input_blocks):
        h = run(m, h, emb)
        if i in feats:
            h = h + feats[i].type(h.dtype)
        hs.append(h)
    h = run(adm.middle_block, h, emb)
    for m in adm.output_blocks:
        h = run(m, torch.cat([h, hs.pop()], 1), emb)
    return adm.out(h.type(x.dtype))[:, :3].float()


INJECT = injection_points()
adapter = JEPAAdapter(jepa.embed_dim, INJECT, adm.model_channels * 4).to(D0)
print("injection points {block: (px, channels)}:", INJECT, f"| adapter {sum(p.numel() for p in adapter.parameters()) / 1e6:.1f}M params")

BETAS = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64)  # ADM's linear schedule
ABAR = torch.cumprod(1 - BETAS, 0)
ABAR_D = ABAR.float().to(D0)

with torch.no_grad():  # the re-implemented forward equals ADM's, and the untrained adapter changes nothing
    xt, tt = torch.randn(2, 3, S, S, device=D0), torch.tensor([10, 500], device=D0)
    ref = adm(xt, tt)[:, :3].float()
    assert torch.allclose(eps_pred(xt, tt), ref, atol=1e-3)
    assert torch.allclose(eps_pred(xt, tt, adapter(jepa_latents(EVAL_CLEAN[:2]))), ref, atol=1e-3)


found = find_input("jepa-guided-diffusion/**/jepa_adapter.pt")
assert found, "attach the output of the jepa-guided-diffusion notebook (version 1) as input data"
adapter.load_state_dict(torch.load(found, map_location="cpu", weights_only=True))
adapter.eval()
def t_for_sigma(sigma):  # pixel σ on [0, 1] -> step whose noise level matches (x in [-1, 1] doubles σ)
    return int((ABAR - 1 / (1 + (2 * sigma) ** 2)).abs().argmin())


def to01(z):
    return ((z + 1) / 2).clamp(0, 1).cpu()

## The two edits, as programs over the trained weights

In [ ]:
# E2: the glob path ignores the latents (h(c) = SiLU(b1) = h(0) for every c)
E2 = copy.deepcopy(adapter)
with torch.no_grad():
    E2.glob[0].weight.zero_()

# E3: rank-1 update of glob.2 that removes the offset common to all latent-conditioned inputs, keeping h(0) fixed
probe = find_input("jepa-adapter-weight-probe/**/latents.pt")
assert probe, "attach the output of the jepa-adapter-weight-probe notebook as input data"
LAT = torch.load(probe, map_location="cpu", weights_only=True)
W1, b1, W2 = (t.detach().double().cpu() for t in (adapter.glob[0].weight, adapter.glob[0].bias, adapter.glob[2].weight))
c_bar = torch.stack([c.float().mean((2, 3))[0].double() for c in LAT.values()])
H = F.silu(c_bar @ W1.T + b1)
h0 = F.silu(b1)
m = W2 @ (H.mean(0) - h0)                                   # mean latent-induced output offset
P_perp = torch.eye(len(h0), dtype=torch.float64) - torch.outer(h0, h0) / (h0 @ h0)
A = H @ P_perp
v = P_perp @ torch.linalg.solve(A.T @ A + E3_RIDGE * torch.eye(len(h0), dtype=torch.float64), A.T @ torch.ones(len(H), dtype=torch.float64))
E3 = copy.deepcopy(adapter)
with torch.no_grad():
    E3.glob[2].weight.copy_((W2 - torch.outer(m, v)).float())

with torch.no_grad():   # both edits leave the null-latent output unchanged
    z = torch.zeros(1, jepa.embed_dim, 16, 16, device=D0)
    ref_f, ref_g = adapter(z)
    for name, edited in (("E2", E2), ("E3", E3)):
        f_e, g_e = edited(z)
        gap = max(float((g_e - ref_g).abs().max()), *(float((f_e[i] - ref_f[i]).abs().max()) for i in ref_f))
        print(f"{name}: max |null output change| {gap:.2e}")
        assert gap < 1e-5
print(f"E3: |m| rms {float(m.pow(2).mean().sqrt()):.4f}, v·h(c) on the probe cases {float((H @ v).mean()):.3f} ± {float((H @ v).std()):.3f}")
torch.save({"E2": E2.state_dict(), "E3": E3.state_dict()}, f"{OUT}/edited_adapters.pt")

## Denoise v1's 52 cases: ADM, original adapter (null and JEPA latents), E2 and E3

In [ ]:
import lpips

LPIPS = lpips.LPIPS(net="alex", verbose=False).to(D0)


def scores(out, clean):
    a, b = out.permute(1, 2, 0).numpy(), clean.permute(1, 2, 0).numpy()
    return dict(psnr=peak_signal_noise_ratio(b, a, data_range=1.0),
                ssim=structural_similarity(b, a, channel_axis=-1, data_range=1.0),
                lpips=LPIPS(out[None].to(D0) * 2 - 1, clean[None].to(D0) * 2 - 1).item())


VARIANTS = [("ADM", None, False), ("null", adapter, False), ("JEPA", adapter, True), ("JEPA E2", E2, True), ("JEPA E3", E3, True)]


@torch.no_grad()
def denoise_all(y, sigma, c, seed=0):
    """v1's denoise_variants for every variant in one batch: reverse process from t*, same start and same noise draws."""
    conds = [None if a is None else a(c if use_c else torch.zeros_like(c)) for _, a, use_c in VARIANTS]
    feats, emb = conds[1]
    cond = ({i: torch.cat([torch.zeros_like(feats[i]) if cd is None else cd[0][i] for cd in conds]) for i in feats},
            torch.cat([torch.zeros_like(emb) if cd is None else cd[1] for cd in conds]))
    V, t0 = len(conds), t_for_sigma(sigma)
    gen = torch.Generator(device=D0).manual_seed(seed)
    x = (ABAR[t0].sqrt().float() * (2 * y.to(D0) - 1))[None].repeat(V, 1, 1, 1)
    tweedie = None
    for t in range(t0, -1, -1):
        a_t, beta_t = ABAR[t].item(), BETAS[t].item()
        a_prev = ABAR[t - 1].item() if t > 0 else 1.0
        eps = eps_pred(x, torch.full((V,), t, device=D0, dtype=torch.long), cond)
        x0 = ((x - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5).clamp(-1, 1)
        if tweedie is None:
            tweedie = x0
        if t == 0:
            x = x0
            break
        mean = (a_prev ** 0.5 * beta_t / (1 - a_t)) * x0 + ((1 - beta_t) ** 0.5 * (1 - a_prev) / (1 - a_t)) * x
        noise = torch.randn((1, 3, S, S), generator=gen, device=D0).expand_as(x)  # same draw for every variant
        x = mean + (beta_t * (1 - a_prev) / (1 - a_t)) ** 0.5 * noise
    return to01(tweedie), to01(x)


rows, KEEP, t_start = [], {}, time.time()
for k, case in enumerate(CASES):
    tw, smp = denoise_all(case["noisy"], case["sigma"], jepa_latents(case["noisy"][None]), seed=k)
    outs = {"noisy input": case["noisy"]}
    for v_, (name, _, _) in enumerate(VARIANTS):
        outs[f"{name}: Tweedie"], outs[f"{name}: reverse"] = tw[v_], smp[v_]
    clean, noise = case["clean"], case["noisy"] - case["clean"]
    for m_, o in outs.items():
        err = o - clean
        rows.append(dict(image=case["image"], noise=case["noise"], method=m_, kept=float((err * noise).sum() / (noise * noise).sum()),
                         **scores(o, clean)))
    if case["save"]:
        KEEP[(case["image"], case["noise"])] = (clean, outs)
        folder = f"{OUT}/images/{slug(case['image'])}__{slug(case['noise'])}"
        os.makedirs(folder, exist_ok=True)
        for j, (name, img) in enumerate([("clean (reference)", clean), *outs.items()]):
            save_png(img, f"{folder}/{j:02d}-{slug(name)}.png")
    if k % 6 == 0 or k == len(CASES) - 1:
        print(f"  case {k + 1}/{len(CASES)}  {(time.time() - t_start) / 60:.1f} min", flush=True)
res = pd.DataFrame(rows)
res.to_csv(f"{OUT}/scores.csv", index=False)

## Results

In [ ]:
METHODS = list(dict.fromkeys(res.method))
for metric in ("psnr", "lpips", "kept"):
    print(f"\nmean {metric}")
    print(res.pivot_table(index="method", columns="noise", values=metric).reindex(METHODS).round(4).to_string())
merged = res.merge(ORIGINAL.replace({"method": {"ADM Tweedie": "ADM: Tweedie", "ADM reverse": "ADM: reverse",
                                                 "adapter null: reverse": "null: reverse", "adapter JEPA: reverse": "JEPA: reverse",
                                                 "adapter JEPA: Tweedie": "JEPA: Tweedie"}}),
                   on=["image", "noise", "method"], suffixes=("", "_v1"))
RERUN_GAP = {m_: float((g.psnr - g.psnr_v1).abs().max()) for m_, g in merged.groupby("method")}
print("\nre-run vs v1 on identical inputs and seeds, max |Δ PSNR| (dB):", {k: round(v, 4) for k, v in RERUN_GAP.items()})

wide = res.pivot_table(index=["noise", "image"], columns="method", values=["psnr", "kept"])
print("\nper-image Δ PSNR and Δ kept against null (reverse): mean, worst image")
for name in ("JEPA", "JEPA E2", "JEPA E3"):
    for noise in dict.fromkeys(res.noise):
        dp = (wide["psnr"][f"{name}: reverse"] - wide["psnr"]["null: reverse"]).loc[noise]
        dk = (wide["kept"][f"{name}: reverse"] - wide["kept"]["null: reverse"]).loc[noise]
        print(f"  {name:8s} {noise:13s} Δ PSNR mean {dp.mean():+.3f} worst {dp.min():+.3f} ({dp.idxmin()}) | Δ kept mean {dk.mean():+.4f} worst {dk.max():+.4f} ({dk.idxmax()})")

fig, axs = plt.subplots(1, 2, figsize=(13, 4.3))
noises = list(dict.fromkeys(res.noise))
for a, metric, label in zip(axs, ("psnr", "kept"), ("Δ PSNR vs null, dB (reverse)", "Δ share of input noise kept vs null (reverse)")):
    for shift, name, color in ((-0.22, "JEPA", "tab:red"), (0.0, "JEPA E2", "tab:green"), (0.22, "JEPA E3", "tab:blue")):
        d = wide[metric][f"{name}: reverse"] - wide[metric]["null: reverse"]
        for p, noise in enumerate(noises):
            vals = d.loc[noise].to_numpy()
            a.scatter(p + shift + np.linspace(-0.08, 0.08, len(vals)), vals, s=14, color=color, label=name if p == 0 else None)
    a.axhline(0, color="black", lw=0.8)
    a.set_xticks(range(len(noises))); a.set_xticklabels(noises); a.set_ylabel(label); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/per_image.png", dpi=110); plt.show()

key = ("div2k_9", "gaussian 0.2")
if key in KEEP:
    clean, outs = KEEP[key]
    panels = [("clean", clean), ("noisy input", outs["noisy input"])] + [(f"{n}: reverse", outs[f"{n}: reverse"]) for n, _, _ in VARIANTS]
    shown = res.set_index(["image", "noise", "method"])
    fig, axs = plt.subplots(2, len(panels), figsize=(2.4 * len(panels), 5.0))
    for j, (name, img) in enumerate(panels):
        title = name if j == 0 else f"{name}\n{shown.loc[(*key, name)].psnr:.2f} dB | kept {shown.loc[(*key, name)].kept:.3f}"
        axs[0, j].imshow(img.permute(1, 2, 0).numpy()); axs[0, j].set_title(title, fontsize=7)
        axs[1, j].imshow(img[:, 64:192, 64:192].permute(1, 2, 0).numpy()); axs[1, j].set_title("zoom", fontsize=7)
        for a in axs[:, j]:
            a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.savefig(f"{OUT}/div2k_9_sigma_0.2.png", dpi=100); plt.show()

with open(f"{OUT}/summary.json", "w") as f:
    json.dump(dict(rerun_gap_db=RERUN_GAP, input_psnr_gap_db=INPUT_GAP, e3_ridge=E3_RIDGE,
                   mean=res.groupby(["noise", "method"], sort=False)[["psnr", "ssim", "lpips", "kept"]].mean().reset_index().to_dict("records")),
              f, indent=1, default=float)
open(f"{OUT}/DONE", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
print(sorted(os.listdir(OUT)))